# Optimization of a Recycling Mixer-Reactor-Clarifier Activated Sludge System Using a Physically-Constrained Statistical Surrogate

This notebook is the staged executable companion to `article/wip_v2/manuscript.tex`. It uses the five-reactor ASM2d-TSN model, ten-layer Clarifier, independent development/calibration/assessment designs, a 170-response statistical surrogate, and the single combined physics-constrained statistical NLP documented there.

Every section is an independent gate. A failed gate stops execution; rejected mechanistic rows are retained for diagnosis and are never replaced.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

from closed_loop.workflow import ClosedLoopWorkflow

REPOSITORY_ROOT = Path.cwd().resolve()
CONFIG_PATH = REPOSITORY_ROOT / 'config' / 'params_closed_loop.json'
PROFILE = os.environ.get('CLOSED_LOOP_PROFILE', 'test_2000')
RUN_ID = os.environ.get('CLOSED_LOOP_RUN_ID')
if not RUN_ID:
    raise RuntimeError('Set CLOSED_LOOP_RUN_ID to a new immutable run identifier before execution.')
COMPLETION_PATH = REPOSITORY_ROOT / 'results' / 'closed_loop' / RUN_ID / 'COMPLETED.json'
if COMPLETION_PATH.is_file():
    completed = json.loads(COMPLETION_PATH.read_text(encoding='utf-8'))
    raise RuntimeError(
        f"Run {RUN_ID!r} is already sealed with status {completed.get('status')!r}; "
        'inspect its artifacts or choose a new run identifier.'
    )

workflow = ClosedLoopWorkflow(
    config_path=CONFIG_PATH,
    profile=PROFILE,
    run_id=RUN_ID,
    repository_root=REPOSITORY_ROOT,
)
print(json.dumps({'profile': PROFILE, 'run_id': RUN_ID, 'run_root': str(workflow.run_root)}, indent=2))

## 1. Static mathematical and design contract

This gate reconstructs and audits the 28-by-20 stoichiometric matrix and five invariant rows, checks every configured dimension and coordinate order, generates the independent SplitMix64 development, calibration, and assessment blocks, and freezes their membership.

In [ ]:
manifest = workflow.run(through='static')
manifest['stages']['static']

## 2. Mechanistic pilots and frequent generation checks

The first 256 immutable design rows are solved in checkpoints ending at 4, 16, 64, and 256 rows. Every checkpoint verifies the full steady-state, external-balance, positivity, Clarifier, and local-stability contract before more rows are attempted.

In [ ]:
manifest = workflow.run(through='pilot')
manifest['stages']['pilot']

## 3. Complete mechanistic design

Generation resumes from the sealed pilot chunks and proceeds through every configured checkpoint. For `test_2000`, this produces exactly 2,000 accepted 110-state steady solutions and their 170-coordinate targets. The `full` profile instead produces the article's independent 20,000-point design.

In [ ]:
manifest = workflow.run(through='dataset')
manifest['stages']['dataset']

## 4. Development-only fit

Only the 70% development block determines centers, scales, the 351-feature OLS coefficients, smooth-equation scales, and numerical audits. No calibration, assessment, or optimization row can enter this fit, and no later production refit is permitted.

In [ ]:
manifest = workflow.run(through='fit')
manifest['stages']['fit']

## 5. Calibration and untouched assessment

The independent 10% block fixes the 95% split-conformal fidelity radius. The final 20% block is then evaluated once against the frozen model and threshold; it cannot change any coefficient, scale, tolerance, or downstream setting.

In [ ]:
manifest = workflow.run(through='calibration')
manifest = workflow.run(through='assessment')
{'calibration': manifest['stages']['calibration'], 'assessment': manifest['stages']['assessment']}

## 6. NLP preflight, nominal, robustness, and sensitivity cases

Each case uses nine deterministic CasADi--IPOPT starts for one combined 115-variable NLP. Its 110 physical states satisfy the smooth mechanistic steady-state equations, while the reconstructed 170-coordinate response must satisfy the frozen conformal-fidelity and leverage limits. The selected local candidate receives one independent exact nonsmoothed BDF replay. Exact fidelity uses the NLP's normalized scale: `d_BDF / delta - 1` must not exceed the configured normalized feasibility tolerance (currently `1e-8`). On resume, completed per-start and exact-replay artifacts must pass their identity and digest checks before reuse; verified scientific calls are not repeated, and persisted invocation records supply realized counts. The `test_2000` profile executes the nominal case, ten robustness cases, and all twelve sensitivity cases: 207 NLP starts and at most 2,023 total BDF routes including design generation. Physical projection QPs and DIRECT searches are not part of this route.

In [ ]:
manifest = workflow.run(through='nlp_preflight')
manifest = workflow.run(through='optimization')
manifest['stages']['optimization']

## 7. Reports and immutable completion seal

Tables and figures are generated from row-level artifacts. The `report` stage writes an immutable summary with `release_status="provisional_pending_terminal_replay"` and `release_authority="COMPLETED.json is created only after terminal replay and sealing"`. Terminal acceptance reloads the stored unrounded state, replays the declared numerical and scientific checks, verifies every stage marker, and writes the completion seal and `COMPLETED.json`; its `report_release_status="terminally_sealed"` records release authorization. Only that sealed record is final scientific output, and a sealed run identifier cannot be reused.

In [ ]:
manifest = workflow.run(through='report')
manifest = workflow.run(through='complete')
summary = {
    'run_id': manifest['run_id'],
    'profile': manifest['profile'],
    'article_eligible': manifest['article_eligible'],
    'status': manifest['status'],
    'completed': (workflow.run_root / 'COMPLETED.json').is_file(),
}
print(json.dumps(summary, indent=2))
summary